In [2]:
pip install tensorflow

     ---------------------------------------- 0.0/49.8 kB ? eta -:--:--
     ---------------------------------------- 49.8/49.8 kB 2.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/376.0 MB ? eta -:--:--
   ---------------------------------------- 0.4/376.0 MB 9.2 MB/s eta 0:00:42
   ---------------------------------------- 2.0/376.0 MB 20.8 MB/s eta 0:00:18
   ---------------------------------------- 3.8/376.0 MB 30.1 MB/s eta 0:00:13
    --------------------------------------- 7.3/376.0 MB 38.7 MB/s eta 0:00:10
   - -------------------------------------- 10.5/376.0 MB 50.4 MB/s eta 0:00:08
   - -------------------------------------- 13.8/376.0 MB 72.6 MB/s eta 0:00:05
   - -------------------------------------- 17.0/376.0 MB 72.6 MB/s eta 0:00:05
   -- ------------------------------------- 19.8/376.0 MB 81.8 MB/s eta 0:00:05
   -- ------------------------------------- 23.5/376.0 MB 72.6 MB/s eta 0:00:05
   -- ------------------------------------- 25.9/376.0 MB 65.2


[notice] A new release of pip is available: 24.0 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import cv2
import numpy as np
import mediapipe as mp
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import Layer, Conv2D, BatchNormalization, Activation, Dropout, Add, Input, GlobalAveragePooling2D, Dense, GaussianNoise
import math

# ─── Helper: Normalize Adjacency ────────────────────────────────────────────────
def normalize_adjacency(A):
    I = np.eye(len(A))
    A_hat = A + I
    D = np.sum(A_hat, axis=1)
    D_inv_sqrt = np.diag(1.0 / np.sqrt(D))
    return D_inv_sqrt @ A_hat @ D_inv_sqrt

# ─── ST-GCN Block ───────────────────────────────────────────────────────────────
class STGCNBlock(Layer):
    def __init__(self, out_channels, adjacency, t_kernel_size=9, stride=1, dropout_rate=0.3, **kwargs):
        super().__init__(**kwargs)
        self.out_channels = out_channels
        self.A = np.array(adjacency)
        self.A_norm = tf.constant(normalize_adjacency(self.A), dtype=tf.float32)
        self.t_kernel_size = t_kernel_size
        self.stride = stride
        self.conv_s = Conv2D(filters=out_channels, kernel_size=(1,1), padding='valid')
        self.conv_t = Conv2D(filters=out_channels, kernel_size=(t_kernel_size,1),
                             strides=(stride,1), padding='same')
        self.bn     = BatchNormalization()
        self.act    = Activation('relu')
        self.dropout= Dropout(dropout_rate)
        self.skip_conv = None

    def build(self, input_shape):
        batch, T, N, C = input_shape
        self.conv_s.build(input_shape)
        shape_s = self.conv_s.compute_output_shape(input_shape)
        self.conv_t.build(shape_s)
        shape_t = self.conv_t.compute_output_shape(shape_s)
        self.bn.build(shape_t)
        if C != self.out_channels or self.stride != 1:
            self.skip_conv = Conv2D(filters=self.out_channels, kernel_size=(1,1),
                                    strides=(self.stride,1), padding='same')
            self.skip_conv.build(input_shape)
        super().build(input_shape)

    def call(self, x, training=False):
        b, T, N, C = tf.shape(x)[0], tf.shape(x)[1], tf.shape(x)[2], tf.shape(x)[3]
        x_ = tf.reshape(x, [b * T, N, C])
        x_ = tf.matmul(self.A_norm, x_)
        x_ = tf.reshape(x_, [b, T, N, C])
        x_ = self.conv_s(x_)
        x_ = self.conv_t(x_)
        x_ = self.bn(x_, training=training)
        skip = x
        if self.skip_conv:
            skip = self.skip_conv(skip)
        x_ = Add()([x_, skip])
        x_ = self.act(x_)
        return self.dropout(x_, training=training)

    def get_config(self):
        config = super().get_config()
        config.update({
            'out_channels': self.out_channels,
            'adjacency': self.A.tolist(),
            't_kernel_size': self.t_kernel_size,
            'stride': self.stride
        })
        return config

# ─── CONSTANTS ─────────────────────────────────────────────────────────────────
SEQ_LEN   = 30
NUM_POSE  = 33
NUM_HAND  = 21
NUM_NODES = NUM_POSE + NUM_HAND * 2
COORD_DIM = 3
PAD_VAL   = -1.0

# ─── LOAD ST-GCN MODEL ─────────────────────────────────────────────────────────
model = load_model(
    'stgcn_model.keras',
    custom_objects={'STGCNBlock': STGCNBlock}
)

# ─── MEDIAPIPE SETUP ───────────────────────────────────────────────────────────
mp_pose  = mp.solutions.pose
mp_hands = mp.solutions.hands
mp_draw  = mp.solutions.drawing_utils

pose = mp_pose.Pose(static_image_mode=False,
                    min_detection_confidence=0.5,
                    min_tracking_confidence=0.5)
hands = mp_hands.Hands(static_image_mode=False,
                       max_num_hands=2,
                       min_detection_confidence=0.5,
                       min_tracking_confidence=0.5)

# ─── RUN WEBCAM LOOP ────────────────────────────────────────────────────────────
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise IOError("Cannot open webcam")

sequence_buffer = []

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)
    rgb   = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    res_p = pose.process(rgb)
    res_h = hands.process(rgb)

    # draw for debug
    if res_p.pose_landmarks:
        mp_draw.draw_landmarks(frame, res_p.pose_landmarks, mp_pose.POSE_CONNECTIONS)
    if res_h.multi_hand_landmarks:
        for h in res_h.multi_hand_landmarks:
            mp_draw.draw_landmarks(frame, h, mp_hands.HAND_CONNECTIONS)

    # ─── extract landmarks per frame ───────────────────────────────────────────
    lm_frame = []
    # pose
    if res_p.pose_landmarks:
        for lm in res_p.pose_landmarks.landmark:
            lm_frame.append([lm.x, lm.y, lm.z])
    else:
        lm_frame.extend([[PAD_VAL, PAD_VAL, PAD_VAL]] * NUM_POSE)
    # hands
    hand_pts = [[PAD_VAL]*3]*(NUM_HAND*2)
    if res_h.multi_hand_landmarks:
        for i, hand_landmarks in enumerate(res_h.multi_hand_landmarks):
            start = i * NUM_HAND
            for j, lm in enumerate(hand_landmarks.landmark):
                hand_pts[start + j] = [lm.x, lm.y, lm.z]
    lm_frame.extend(hand_pts)

    sequence_buffer.append(lm_frame)
    if len(sequence_buffer) > SEQ_LEN:
        sequence_buffer.pop(0)

    # ── when buffer full, predict ──────────────────────────────────────────────
    if len(sequence_buffer) == SEQ_LEN:
        arr = np.array(sequence_buffer, dtype=np.float32)     # (30,75,3)
        inp = np.expand_dims(arr, axis=0)                     # (1,30,75,3)
        preds = model.predict(inp, verbose=0)[0]
        cls   = np.argmax(preds)
        conf  = preds[cls]

        cv2.putText(frame, f"Class: {cls} ({conf:.2f})",
                    (10,30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 2)

    cv2.imshow("ST-GCN Inference", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
pose.close()
hands.close()



c:\Users\arjun\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\saving\saving_lib.py:757: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 40 variables whereas the saved optimizer has 78 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


ValueError: Input 0 of layer "functional_35" is incompatible with the layer: expected shape=(None, 30, 55, 3), found shape=(1, 30, 75, 3)

In [10]:
import cv2
import numpy as np
import mediapipe as mp
from tensorflow.keras.models import load_model

# ─── Load model and dynamically get its input shape ─────────────────────────
model = load_model('stgcn_model.keras', custom_objects={'STGCNBlock': STGCNBlock})
_, SEQ_LEN, NUM_NODES, COORD_DIM = model.input_shape
print(f"Expecting input (1,{SEQ_LEN},{NUM_NODES},{COORD_DIM})")

PAD_VAL = -1.0

# ─── Mediapipe setup ────────────────────────────────────────────────────────
mp_pose  = mp.solutions.pose
mp_hands = mp.solutions.hands
mp_draw  = mp.solutions.drawing_utils

pose = mp_pose.Pose(static_image_mode=False,
                    min_detection_confidence=0.5,
                    min_tracking_confidence=0.5)
hands = mp_hands.Hands(static_image_mode=False,
                       max_num_hands=2,
                       min_detection_confidence=0.5,
                       min_tracking_confidence=0.5)

# ─── Webcam + sliding window ────────────────────────────────────────────────
cap = cv2.VideoCapture(0)
sequence_buffer = []

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)
    rgb   = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    res_p = pose.process(rgb)
    res_h = hands.process(rgb)

    # Draw for debug
    if res_p.pose_landmarks:
        mp_draw.draw_landmarks(frame, res_p.pose_landmarks, mp_pose.POSE_CONNECTIONS)
    if res_h.multi_hand_landmarks:
        for h in res_h.multi_hand_landmarks:
            mp_draw.draw_landmarks(frame, h, mp_hands.HAND_CONNECTIONS)

    # ─── Extract all 75 landmarks ────────────────────────────────────────────
    lm_frame = []
    # Pose (33)
    if res_p.pose_landmarks:
        for lm in res_p.pose_landmarks.landmark:
            lm_frame.append([lm.x, lm.y, lm.z])
    else:
        lm_frame.extend([[PAD_VAL]*3] * 33)

    # Hands (42)
    hand_pts = [[PAD_VAL]*3] * 42
    if res_h.multi_hand_landmarks:
        for i, hand_landmarks in enumerate(res_h.multi_hand_landmarks):
            start = i * 21
            for j, lm in enumerate(hand_landmarks.landmark):
                hand_pts[start + j] = [lm.x, lm.y, lm.z]
    lm_frame.extend(hand_pts)

    # ─── Trim to the first NUM_NODES landmarks ───────────────────────────────
    lm_frame = lm_frame[:NUM_NODES]      # now length == NUM_NODES

    sequence_buffer.append(lm_frame)
    if len(sequence_buffer) > SEQ_LEN:
        sequence_buffer.pop(0)

    # ─── When buffer full, predict ──────────────────────────────────────────
    if len(sequence_buffer) == SEQ_LEN:
        arr = np.array(sequence_buffer, dtype=np.float32)   # (30, NUM_NODES, 3)
        inp = np.expand_dims(arr, axis=0)                   # (1, 30, NUM_NODES, 3)
        preds = model.predict(inp, verbose=0)[0]
        cls   = np.argmax(preds)
        conf  = preds[cls]
        cv2.putText(frame, f"Class: {cls} ({conf:.2f})",
                    (10,30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 2)

    cv2.imshow("ST-GCN Inference", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
pose.close()
hands.close()


Expecting input (1,30,55,3)


KeyboardInterrupt: 